In [ ]:
import nibabel as nib
import numpy as np
from model import UNet
from helper import get_cache, normalized_modality, format_index, get_filepath, pad_to_256, diagnose_timing
from inference import infer
from torch.utils.data import random_split, DataLoader, Dataset
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn 
import torch
import wandb
from monai.losses import DiceCELoss
from pathlib import Path
from dotenv import dotenv_values, find_dotenv

config = dotenv_values(find_dotenv(usecwd=True))
TR_DATA_PATH = Path(config.get("TR_DATA_PATH"))

BRAINS = int(config.get("NUM_BRAINS"))
wandb.login(config.get("WANDB_API_KEY"));

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: C:\Users\kacpe\_netrc


In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [ ]:
# class CustomDataset(Dataset):
#     def __init__(self, path):
#         super().__init__()
#         self.path = path
#         self.dirnames = [d for d in Path(path).iterdir() if d.is_dir()]
#         self.len = BRAINS
#         # self.len = len(self.dirnames) # 369 brains
#         self.slices = 155 # slices per brain

#     def __len__(self):
#         return self.len
            
#     def __getitem__(self, index):
#         # t1 = self.get_brain_mod(index, "t1")
#         t1ce = self.get_brain_mod(index, "t1ce")
#         # t2 = self.get_brain_mod(index, "t2")
#         flair = self.get_brain_mod(index, "flair")
#         seg = self.get_brain_mod(index, "seg")
#         brain = torch.cat([flair, t1ce], dim=1)
#         return brain, seg
    
#     def __getitem__(self, index):
#         brain_index = index + 1

#         # load from cache instead of raw NIfTI
#         flair = preprocess_and_cache(brain_index, "flair")  # (155, 1, 128, 128)
#         t1ce  = preprocess_and_cache(brain_index, "t1ce")   # (155, 1, 128, 128)
#         seg   = preprocess_and_cache(brain_index, "seg")    # (155, 128, 128)

#         # filter out empty slices
#         has_tumor = seg.amax(dim=(1, 2)) > 0   # slices with any tumor voxel
#         rand_bg   = torch.rand(self.slices) < 0.2  # keep 20% of background slices
#         keep      = has_tumor | rand_bg

#         flair = flair[keep]   # (N, 1, 128, 128)
#         t1ce  = t1ce[keep]    # (N, 1, 128, 128)
#         seg   = seg[keep]     # (N, 128, 128)
#

#         brain = torch.cat([flair, t1ce], dim=1)  # (N, 2, 128, 128)
#         return brain, seg
        

In [12]:
class CustomDataset(Dataset):
    def __init__(self, path):
        super().__init__()
        self.data = []
        print("Loading all brains into RAM...")
        for index in range(BRAINS):
            brain_index = index + 1
            flair = get_cache(brain_index, "flair")
            t1ce  = get_cache(brain_index, "t1ce")
            seg   = get_cache(brain_index, "seg")
            self.data.append((flair, t1ce, seg))
            print(f"Loaded brain {brain_index}/{BRAINS}", end="\r")
        print("\nAll brains loaded.")

    def __len__(self):
        return len(self.data)
        
    def __getitem__(self, index):
        flair, t1ce, seg = self.data[index]

        has_tumor = seg.amax(dim=(1, 2)) > 0
        rand_bg   = torch.rand(seg.shape[0]) < 0.2
        keep      = has_tumor | rand_bg

        brain = torch.cat([flair[keep], t1ce[keep]], dim=1)
        return brain, seg[keep]

In [13]:
train_dataset = CustomDataset(TR_DATA_PATH)

train_size = int(len(train_dataset) * 0.9)
val_size = len(train_dataset) - train_size

train_sub, val_sub = random_split(train_dataset, [train_size, val_size])
train_loader = DataLoader(train_sub, batch_size=None, shuffle=True, pin_memory=True)

val_loader = DataLoader(val_sub, batch_size=None, shuffle=False, pin_memory=True)

Loading all brains into RAM...
Loaded brain 120/120
All brains loaded.


In [14]:
CLASS_NAMES = ["background", "necrotic", "edema", "enhancing"]

In [15]:
def dice_per_class(logits, y, num_classes=4, eps=1e-6):
    preds = logits.argmax(dim=1)
    scores = {}

    for c in range(num_classes):
        pred_c = (preds == c).float()
        true_c = (y == c).float()

        intersection = (pred_c * true_c).sum()
        denom = pred_c.sum() + true_c.sum()

        # if a class is absent in both pred and target, score is 1 (perfect)
        if denom < eps:
            scores[CLASS_NAMES[c]] = 1.0
        else:
            scores[CLASS_NAMES[c]] = (2.0 * intersection / denom).item()

    return scores

In [16]:
@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    total_samples = 0
    cumulative_dice = {c: 0.0 for c in CLASS_NAMES}
    num_batches = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        logits = model(x)
        loss = criterion(logits, y)#.unsqueeze(1)) # needed for loss function from monai

        total_loss += loss.item() * x.size(0)
        total_samples += x.size(0)

        batch_dice = dice_per_class(logits, y)
        for c in CLASS_NAMES:
            cumulative_dice[c] += batch_dice[c]
        num_batches += 1

    avg_dice = {c: cumulative_dice[c] / num_batches for c in CLASS_NAMES}
    return total_loss / total_samples, avg_dice

In [17]:
def train_one_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0.0
    total_samples = 0
    cumulative_dice = {c: 0.0 for c in CLASS_NAMES}
    num_batches = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        yb_input = y#.unsqueeze(1) #needed for the loss function from monai.

        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, yb_input)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)
        total_samples += x.size(0)

        batch_dice = dice_per_class(logits.detach(), y)
        for c in CLASS_NAMES:
            cumulative_dice[c] += batch_dice[c]
        num_batches += 1
        
    avg_dice = {c: cumulative_dice[c] / num_batches for c in CLASS_NAMES}
    return total_loss / total_samples, avg_dice

In [19]:
model = UNet()
model = model.to(device)

In [ ]:
LR = 5e-4
EPOCHS = 40

optimizer = optim.Adam(model.parameters(), lr = LR)

# criterion = DiceFocalLoss(
#     to_onehot_y=True,
#     softmax=True,
#     squared_pred=True,
#     gamma=2.0,       # focal strength, the higher it is the more it focuses on hard examples
#     lambda_dice=0.5, # balance between Dice and Focal terms
#     lambda_focal=0.5,
# )

class_weights = torch.tensor([0.1, 2.0, 2.0, 2.0], device=device)
criterion = nn.CrossEntropyLoss(
    weight=class_weights
)
# criterion = DiceCELoss(
#     to_onehot_y=True,
#     softmax=True,
#     squared_pred=True,
#     weight=class_weights,
# )

run = wandb.init(
    entity="kappo-university-of-wroc-aw",
    project="Brain-tumor-segmentation",
    config={
        "learning_rate": LR,
        "architecture": "UNET",
        "dataset": "BraTS2020",
        "loss" : "weighted nn.CrossEntropy",
        "loss ratio" : "None", # CHANGE THIS TO MATCH THE DICE/ENTOPY RATIO IN THE LOSS FUNCTION WHEN ITS ADDED 
        "weights" : class_weights,
        "epochs": EPOCHS,
        "no. brains": BRAINS,
        "comment": "Loading all the brains at once. Testing showed a big problem in backward computation"
    },
)

for epoch in range(EPOCHS):
    tr_loss, tr_dice = train_one_epoch(model, train_loader, optimizer, criterion, device)
    val_loss, val_dice = evaluate(model, val_loader, criterion, device)

    log_dict = {
        "epoch": epoch + 1,
        "train_loss": tr_loss,
        "val_loss": val_loss,
    }
    
    for c in CLASS_NAMES:
        log_dict[f"train_dice/{c}"] = tr_dice[c]
        log_dict[f"val_dice/{c}"]   = val_dice[c]

    run.log(log_dict)

    print(
        f"Epoch {epoch+1} | "
        f"train_loss={tr_loss:.4f} | val_loss={val_loss:.4f}\n"
        f"  Train Dice: { {k: f'{v:.3f}' for k,v in tr_dice.items()} }\n"
        f"  Val   Dice: { {k: f'{v:.3f}' for k,v in val_dice.items()} }"
    )
run_number = int(run.name.split("-")[-1])
run.finish()

Epoch 1 | train_loss=0.4544 | val_loss=0.2819
  Train Dice: {'background': '0.993', 'necrotic': '0.011', 'edema': '0.414', 'enhancing': '0.155'}
  Val   Dice: {'background': '0.994', 'necrotic': '0.242', 'edema': '0.494', 'enhancing': '0.585'}
Epoch 2 | train_loss=0.2317 | val_loss=0.1991
  Train Dice: {'background': '0.992', 'necrotic': '0.036', 'edema': '0.495', 'enhancing': '0.492'}
  Val   Dice: {'background': '0.996', 'necrotic': '0.168', 'edema': '0.608', 'enhancing': '0.688'}
Epoch 3 | train_loss=0.1976 | val_loss=0.1785
  Train Dice: {'background': '0.993', 'necrotic': '0.106', 'edema': '0.515', 'enhancing': '0.577'}
  Val   Dice: {'background': '0.995', 'necrotic': '0.249', 'edema': '0.595', 'enhancing': '0.707'}
Epoch 4 | train_loss=0.1859 | val_loss=0.1607
  Train Dice: {'background': '0.994', 'necrotic': '0.138', 'edema': '0.528', 'enhancing': '0.606'}
  Val   Dice: {'background': '0.995', 'necrotic': '0.419', 'edema': '0.607', 'enhancing': '0.715'}
Epoch 5 | train_loss=0.1

epoch,▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
train_dice/background,▃▁▃▄▄▄▃▄▄▄▄▅▅▃▅▆▄▅▆▄▆▆▆▆▆▆▆▆▇▆▇▇▇▇▇▇▇█▇█
train_dice/edema,▁▃▄▄▄▅▅▅▅▅▆▆▆▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇█▇▇█████
train_dice/enhancing,▁▅▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇▇███████████████
train_dice/necrotic,▁▁▂▂▄▄▅▅▆▇▇▇▇▇▇▇▇▇█▇▇▇▇█▇███▇███████████
train_loss,█▄▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
val_dice/background,▃▆▅▅▆▁▅▃▆▆▇▅▆▆█▆▅▇▄▇▆█▇▅▄▇▇▆▆▃▅▅▆▄█▇▆▇▇▇
val_dice/edema,▁▄▄▄▅▂▅▄▆▆▇▅▆▆▇▆▆▇▅▇▆█▇▅▅█▇▆▇▄▅▆▆▅██▇▇▇▇
val_dice/enhancing,▁▄▅▅▅▆▆▆▇▇▇▇▇▇█▇▇█▇▇██████▇▇▇▇▇█▇▇▇█████
val_dice/necrotic,▂▁▂▄▃▄▆▆▇▇▇▇▇███▇█▇▇███████▇▇█▇█████████
+1,...


In [ ]:
checkpoint = {
    "epoch": EPOCHS,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}

torch.save(checkpoint, f"results/unet_checkpoint_{run_number}.pth")

In [ ]:
infer(device, 13,[val_sub.indices[i] + 1 for i in range(6)])

In [17]:
# import matplotlib.pyplot as plt

# plt.plot(tr_loss_list)
# plt.xlabel("epoch")
# plt.ylabel("loss")
# plt.show()

# plt.plot(tr_acc_list)
# plt.xlabel("epoch")
# plt.ylabel("acc")
# plt.show()

In [18]:
def aggregate_prediction(pred):
    # may be usefull for simplified segmentation
    whole_tumor = (pred > 0)
    tumor_core = np.isin(pred, [1, 3])
    enhancing = (pred == 3)
    return whole_tumor, tumor_core, enhancing
